**Implement and compare efficient convolutional neural network architectures such as MobileNet or EfficientNet and evaluate their performance in terms of accuracy,model size,and inference speed on an image classification dataset.**

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import time
import os

In [3]:
dataset, info = tfds.load(
    "cats_vs_dogs",
    with_info=True,
    as_supervised=True
)

train_data = dataset["train"]

print("Dataset Loaded Successfully!")
print(info)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.6V4Z9N_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Dataset Loaded Successfully!
tfds.core.DatasetInfo(
    name='cats_vs_dogs',
    full_name='cats_vs_dogs/4.0.1',
    description="""
    A large set of images of cats and dogs. There are 1738 corrupted images that are dropped.
    """,
    homepage='https://www.microsoft.com/en-us/download/details.aspx?id=54765',
    data_dir='/root/tensorflow_datasets/cats_vs_dogs/4.0.1',
    file_format=tfrecord,
    download_size=786.67 MiB,
    dataset_size=1.04 GiB,
    features=FeaturesDict({
        'image': Image(shape=(None, None, 3), dtype=uint8),
        'image/filename': Text(shape=(), dtype=string),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'train': <SplitInfo num_examples=23262, num_shards=16>,
    },
  

In [4]:
TOTAL = info.splits["train"].num_examples

train_size = int(0.8 * TOTAL)
val_size = int(0.2 * TOTAL)

train_ds = train_data.take(train_size)
val_ds = train_data.skip(train_size)

print("Train samples:", train_size)
print("Validation samples:", val_size)

Train samples: 18609
Validation samples: 4652


In [5]:
IMG_SIZE = 224
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

train_ds = train_ds.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Pipeline Ready!")

Pipeline Ready!


In [6]:
def build_mobilenet():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    return model


In [7]:
def build_efficientnet():
    base_model = tf.keras.applications.EfficientNetB0(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    return model

In [8]:
def train_and_evaluate(model, name):

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    print("\n==============================")
    print("Training:", name)
    print("==============================")

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=3
    )

    loss, acc = model.evaluate(val_ds, verbose=0)

    print(name, "Validation Accuracy:", acc)

    return acc

In [9]:
def get_model_size(model, filename):
    model.save(filename)

    size_mb = os.path.getsize(filename) / (1024 * 1024)

    return size_mb


In [10]:
def measure_inference_time(model):

    sample_batch = next(iter(val_ds))[0][:1]

    start = time.time()

    for i in range(50):
        model.predict(sample_batch, verbose=0)

    end = time.time()

    avg_time = (end - start) / 50

    return avg_time

In [11]:
mobilenet_model = build_mobilenet()

mobilenet_acc = train_and_evaluate(mobilenet_model, "MobileNetV2")

mobilenet_size = get_model_size(mobilenet_model, "mobilenet_model.h5")

mobilenet_speed = measure_inference_time(mobilenet_model)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training: MobileNetV2
Epoch 1/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 1002s 2s/step - accuracy: 0.9562 - loss: 0.1243 - val_accuracy: 0.9837 - val_loss: 0.0482
Epoch 2/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 1017s 2s/step - accuracy: 0.9842 - loss: 0.0450 - val_accuracy: 0.9845 - val_loss: 0.0456
Epoch 3/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 958s 2s/step - accuracy: 0.9863 - loss: 0.0384 - val_accuracy: 0.9839 - val_loss: 0.0432


MobileNetV2 Validation Accuracy: 0.9838813543319702


In [ ]:
efficientnet_model = build_efficientnet()

efficientnet_acc = train_and_evaluate(efficientnet_model, "EfficientNetB0")

efficientnet_size = get_model_size(efficientnet_model, "efficientnet_model.h5")

efficientnet_speed = measure_inference_time(efficientnet_model)

In [ ]:
print("\n============================================")
print(" FINAL COMPARISON RESULTS (Cats vs Dogs)")
print("============================================\n")

print("Model\t\tAccuracy\tSize(MB)\tInference Time(sec)")
print("------------------------------------------------------------")

print(f"MobileNetV2\t{mobilenet_acc:.4f}\t\t{mobilenet_size:.2f}\t\t{mobilenet_speed:.5f}")

print(f"EfficientNetB0\t{efficientnet_acc:.4f}\t\t{efficientnet_size:.2f}\t\t{efficientnet_speed:.5f}")
